In [1]:
import pandas as pd
import torch
from torch_geometric.data import Data
import os

In [2]:
#map element symbols to atomic numbers
SYMBOL_MAP = {'H': 1, 'C': 6, 'Co': 27, 'N': 7, 'O': 8, 'F': 9, 'Si':14 ,'P':15, 'S':16, 'Cl':17, 'Br': 35}

In [3]:
def load_cation_dataset(excel_path, xyz_folder):
    df = pd.read_excel(excel_path)
    dataset = []

    for _, row in df.iterrows():
        # 1. Parse XYZ File
        xyz_file = os.path.join(xyz_folder, f"{row['filename']}.xyz")
        with open(xyz_file, 'r') as f:
            lines = f.readlines()
            # Skip first 2 lines (atom count and comment)
            coords = []
            atomic_nums = []
            for line in lines[:]:
                parts = line.split()
                if not parts: continue
                atomic_nums.append([SYMBOL_MAP.get(parts[1], 0)])
                coords.append([float(parts[2]), float(parts[3]), float(parts[4])])

        # 2. Extract Descriptors from Excel
        # Example: Using HOMO-LUMO gap and Cobalt charge as features
        dft_feats = [row['homo_lumo_gap'], row['co_charge']]
        
        # 3. Create Node Features (Broadcast global descriptors to each atom)
        # x shape: [num_atoms, 3] -> [Atomic_Num, Gap, Co_Charge]
        x = torch.tensor([ [num[0]] + dft_feats for num in atomic_nums ], dtype=torch.float)
        pos = torch.tensor(coords, dtype=torch.float)
        y = torch.tensor([row['stability_target']], dtype=torch.float)

        data = Data(x=x, pos=pos, y=y)
        dataset.append(data)
    
    return dataset

# Run and save
# my_data = load_cation_dataset('cations.xlsx', './xyz_files')
# torch.save(my_data, 'final_dataset.pt')

In [4]:
xyz_file = os.path.join("xyz_files/org_xyz_files/di_cocp2_h_h.xyz")
with open(xyz_file, 'r') as f:
    lines = f.readlines()
    # Skip first 2 lines (atom count and comment)
    coords = []
    atomic_nums = []
    for line in lines[:]:
        parts = line.split()
        if not parts: continue
        atomic_nums.append([SYMBOL_MAP.get(parts[1], 0)])
        coords.append([float(parts[2]), float(parts[3]), float(parts[4])])


In [6]:
df=pd.read_excel('cobaltocenium_data.xlsx')
df.head()

,Unnamed: 0,Name,Co[CpY][CpY']OH energy (Eh),CoCpY' name,CoCpY'energy (Eh),CpY name,CpY energy (Eh),BDE (kcal/mol),Dipole moment(Debye),EHOMO(Co[CpY][CpY']OH) (Eh),...,ELUMO(CoCpY'OH) (Eh),EHOMO(CpY) (Eh),ELUMO(CpY) (Eh),CoCpY' Hirshfeld charge,CpY Hirshfeld charge,CoCpY' polarizability,CpY polarizability,Hardness (kcal/mol),dCOM (Å),rCo-O (Å)
0,0,Co(Cp-N(CH₃)₂)(Cp-N(CH₃)₂)OH,-2113.503825,N(CH₃)₂,-1785.987408,N(CH₃)₂,-327.488954,17.233174,15.5954,-0.1803,...,-0.0719,-0.1594,-0.0809,0.011840,0.01184,27.53932,27.53932,-39.156000,3.337480,3.856084
1,1,Co(Cp-N(CH₃)₂)(Cp-OCH₃)OH,-2094.048389,OCH₃,-1766.531474,N(CH₃)₂,-327.488954,17.545539,14.9760,-0.1867,...,-0.0727,-0.1594,-0.0809,-0.015190,0.01184,18.01558,27.53932,-39.532500,3.314513,4.150980
2,2,Co(Cp-N(CH₃)₂)(Cp-O(CO)CH₃)OH,-2207.402596,O(CO)CH₃,-1879.885018,N(CH₃)₂,-327.488954,17.961810,18.7135,-0.1849,...,-0.0855,-0.1594,-0.0809,0.071736,0.01184,27.13368,27.53932,-37.085250,3.312872,4.065847
3,3,Co(Cp-N(CH₃)₂)(Cp-P(CH₃)₂)OH,-2400.122408,P(CH₃)₂,-2072.607246,N(CH₃)₂,-327.488954,16.445457,19.9044,-0.1862,...,-0.0814,-0.1594,-0.0809,-0.059590,0.01184,43.11939,27.53932,-37.744125,3.333984,4.033683
4,4,Co(Cp-N(CH₃)₂)(Cp-S(CH₃))OH,-2417.026291,S(CH₃),-2089.514536,N(CH₃)₂,-327.488954,14.308038,14.9101,-0.1868,...,-0.0817,-0.1594,-0.0809,0.019722,0.01184,27.96205,27.53932,-38.089250,3.328151,3.818543


In [7]:
der=['dme','meth','eth','pr2','thl','c6h5','c2h3','c2h','ch3','tch3','c3h7','h','cl','br','f','conhr',
     'conh2','cooch3','coch3','coh','cof','cocl','cf3','ccl3','cbr3','cn','soor','no2','soocf3','ph_p_cl',
     'ph_o_cl','ph_m_cl','ph_o_cf3','ph_m_cf3','ph_p_cf3','ph_p_ch3','ph_p_och3','ph_op_ch3','ph_op_och3',
     'piperidine','paryl2','naryl2']
der_new=[u'N(CH\u2083)\u2082',u'OCH\u2083',u'O(CO)CH\u2083',u'P(CH\u2083)\u2082',u'S(CH\u2083)',u'Ph',u'C\u2082H\u2083',u'C\u2082H',u'CH\u2083',u'C(CH\u2083)\u2083',u'C\u2083H\u2087',
     'H','Cl','Br','F',u'(CO)NHCH\u2083',u'(CO)NH\u2082',u'(CO)OCH\u2083',u'(CO)CH\u2083','(CO)H','(CO)F','(CO)Cl',u'CF\u2083',u'CCl\u2083',u'CBr\u2083','CN',u'SOOCH\u2083',u'NO\u2082',u'SOOCF\u2083',
    'p-PhCl','o-PhCl','m-PhCl',u'o-PhCF\u2083',u'm-PhCF\u2083',u'p-PhCF\u2083',u'p-PhCH\u2083',u'p-PhOCH\u2083',u'op-Ph(CH\u2083)\u2083',u'op-Ph(OCH\u2083)\u2083',u'NC\u2085H\u2081\u2080',u'P(Ph)\u2082',u'N(Ph)\u2082']

In [8]:
di_complex=[]
for i in range(len(der)):
    for j in range(len(der)):
        if j>=i:
            di_complex.append(f"di_cocp2_{der[i]}_{der[j]}")

In [9]:
di_complex

['di_cocp2_dme_dme',
 'di_cocp2_dme_meth',
 'di_cocp2_dme_eth',
 'di_cocp2_dme_pr2',
 'di_cocp2_dme_thl',
 'di_cocp2_dme_c6h5',
 'di_cocp2_dme_c2h3',
 'di_cocp2_dme_c2h',
 'di_cocp2_dme_ch3',
 'di_cocp2_dme_tch3',
 'di_cocp2_dme_c3h7',
 'di_cocp2_dme_h',
 'di_cocp2_dme_cl',
 'di_cocp2_dme_br',
 'di_cocp2_dme_f',
 'di_cocp2_dme_conhr',
 'di_cocp2_dme_conh2',
 'di_cocp2_dme_cooch3',
 'di_cocp2_dme_coch3',
 'di_cocp2_dme_coh',
 'di_cocp2_dme_cof',
 'di_cocp2_dme_cocl',
 'di_cocp2_dme_cf3',
 'di_cocp2_dme_ccl3',
 'di_cocp2_dme_cbr3',
 'di_cocp2_dme_cn',
 'di_cocp2_dme_soor',
 'di_cocp2_dme_no2',
 'di_cocp2_dme_soocf3',
 'di_cocp2_dme_ph_p_cl',
 'di_cocp2_dme_ph_o_cl',
 'di_cocp2_dme_ph_m_cl',
 'di_cocp2_dme_ph_o_cf3',
 'di_cocp2_dme_ph_m_cf3',
 'di_cocp2_dme_ph_p_cf3',
 'di_cocp2_dme_ph_p_ch3',
 'di_cocp2_dme_ph_p_och3',
 'di_cocp2_dme_ph_op_ch3',
 'di_cocp2_dme_ph_op_och3',
 'di_cocp2_dme_piperidine',
 'di_cocp2_dme_paryl2',
 'di_cocp2_dme_naryl2',
 'di_cocp2_meth_meth',
 'di_cocp2_meth_e

In [14]:
df['xyz_filename']=di_complex

In [15]:
df

,Unnamed: 0,Name,Co[CpY][CpY']OH energy (Eh),CoCpY' name,CoCpY'energy (Eh),CpY name,CpY energy (Eh),BDE (kcal/mol),Dipole moment(Debye),EHOMO(Co[CpY][CpY']OH) (Eh),...,ELUMO(CpY) (Eh),CoCpY' Hirshfeld charge,CpY Hirshfeld charge,CoCpY' polarizability,CpY polarizability,Hardness (kcal/mol),dCOM (Å),rCo-O (Å),xyz filename,xyz_filename
0,0,Co(Cp-N(CH₃)₂)(Cp-N(CH₃)₂)OH,-2113.503825,N(CH₃)₂,-1785.987408,N(CH₃)₂,-327.488954,17.233174,15.5954,-0.1803,...,-0.0809,0.011840,0.01184,27.53932,27.53932,-39.156000,3.337480,3.856084,di_cocp2_dme_dme,di_cocp2_dme_dme
1,1,Co(Cp-N(CH₃)₂)(Cp-OCH₃)OH,-2094.048389,OCH₃,-1766.531474,N(CH₃)₂,-327.488954,17.545539,14.9760,-0.1867,...,-0.0809,-0.015190,0.01184,18.01558,27.53932,-39.532500,3.314513,4.150980,di_cocp2_dme_meth,di_cocp2_dme_meth
2,2,Co(Cp-N(CH₃)₂)(Cp-O(CO)CH₃)OH,-2207.402596,O(CO)CH₃,-1879.885018,N(CH₃)₂,-327.488954,17.961810,18.7135,-0.1849,...,-0.0809,0.071736,0.01184,27.13368,27.53932,-37.085250,3.312872,4.065847,di_cocp2_dme_eth,di_cocp2_dme_eth
3,3,Co(Cp-N(CH₃)₂)(Cp-P(CH₃)₂)OH,-2400.122408,P(CH₃)₂,-2072.607246,N(CH₃)₂,-327.488954,16.445457,19.9044,-0.1862,...,-0.0809,-0.059590,0.01184,43.11939,27.53932,-37.744125,3.333984,4.033683,di_cocp2_dme_pr2,di_cocp2_dme_pr2
4,4,Co(Cp-N(CH₃)₂)(Cp-S(CH₃))OH,-2417.026291,S(CH₃),-2089.514536,N(CH₃)₂,-327.488954,14.308038,14.9101,-0.1868,...,-0.0809,0.019722,0.01184,27.96205,27.53932,-38.089250,3.328151,3.818543,di_cocp2_dme_thl,di_cocp2_dme_thl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
898,898,Co(Cp-NC₅H₁₀)(Cp-P(Ph)₂)OH,-2900.366964,P(Ph)₂,-2456.096523,NC₅H₁₀,-444.238140,20.268403,16.8101,-0.1883,...,-0.0768,-0.040350,-0.08196,150.64300,55.69998,-38.246125,3.322861,4.100101,di_cocp2_piperidine_paryl2,di_cocp2_piperidine_paryl2
899,899,Co(Cp-NC₅H₁₀)(Cp-N(Ph)₂)OH,-2613.765583,NC₅H₁₀,-1902.733250,N(Ph)₂,-710.996495,22.488582,18.9105,-0.1909,...,-0.0954,-0.081960,0.00779,55.69998,133.04620,-40.913000,3.335112,4.585480,di_cocp2_piperidine_naryl2,di_cocp2_piperidine_naryl2
900,900,Co(Cp-P(Ph)₂)(Cp-P(Ph)₂)OH,-3453.710285,P(Ph)₂,-2456.096523,P(Ph)₂,-997.586416,17.159446,17.4096,-0.1905,...,-0.1163,-0.040350,-0.04035,150.64300,150.64300,-34.512500,3.320261,3.993505,di_cocp2_paryl2_paryl2,di_cocp2_paryl2_paryl2
901,901,Co(Cp-P(Ph)₂)(Cp-N(Ph)₂)OH,-3167.111773,P(Ph)₂,-2456.096523,N(Ph)₂,-710.996495,11.768670,21.6211,-0.1909,...,-0.0954,-0.040350,0.00779,150.64300,133.04620,-37.399000,3.327687,4.593649,di_cocp2_paryl2_naryl2,di_cocp2_paryl2_naryl2


In [19]:
df[['Name','xyz_filename']]

,Name,xyz_filename
0,Co(Cp-N(CH₃)₂)(Cp-N(CH₃)₂)OH,di_cocp2_dme_dme
1,Co(Cp-N(CH₃)₂)(Cp-OCH₃)OH,di_cocp2_dme_meth
2,Co(Cp-N(CH₃)₂)(Cp-O(CO)CH₃)OH,di_cocp2_dme_eth
3,Co(Cp-N(CH₃)₂)(Cp-P(CH₃)₂)OH,di_cocp2_dme_pr2
4,Co(Cp-N(CH₃)₂)(Cp-S(CH₃))OH,di_cocp2_dme_thl
...,...,...
898,Co(Cp-NC₅H₁₀)(Cp-P(Ph)₂)OH,di_cocp2_piperidine_paryl2
899,Co(Cp-NC₅H₁₀)(Cp-N(Ph)₂)OH,di_cocp2_piperidine_naryl2
900,Co(Cp-P(Ph)₂)(Cp-P(Ph)₂)OH,di_cocp2_paryl2_paryl2
901,Co(Cp-P(Ph)₂)(Cp-N(Ph)₂)OH,di_cocp2_paryl2_naryl2
